# 1. Setup
**Baseline 2: Multi-task CNN.** One shared backbone with two heads, one for crop and one for disease.
The shared features must serve both tasks. A pair score is log P(crop) + log P(disease).

In [1]:
import copy
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # lets the notebook import common/

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

from common import config
from common.config import CHECKPOINT_DIR, DEVICE, NUM_CROPS, NUM_DISEASES, set_seed
from common.dataset import get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

MODEL_NAME = "baseline2_multitask"
DEBUG = True  # True: small data, 1 epoch. Set False for the full run.
HP = {"backbone": "resnet50", "lr": 1e-4, "weight_decay": 1e-4, "epochs": 20,
      "batch_size": 64, "optimizer": "AdamW", "label_smoothing": 0.1, "disease_loss_weight": 1.0}
EPOCHS = 1 if DEBUG else HP["epochs"]
# Single-process loading avoids Windows worker memory errors.
config.NUM_WORKERS = 0
HP["batch_size"] = 16 if DEBUG else HP["batch_size"]
HP["debug"] = DEBUG
RUN_NAME = MODEL_NAME + ("_debug" if DEBUG else "")
set_seed(42)

# 2. Data

In [2]:
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
    HP["batch_size"], norm="imagenet", debug=DEBUG)
# Reduce host memory pressure before any loader is iterated.
for loader in (train_loader, val_loader, seen_loader, unseen_loader):
    loader.pin_memory = False
pairs = get_pairs().to(DEVICE)

# 3. Model

In [3]:
class MultiTaskNet(nn.Module):
    """Shared backbone, two linear heads."""

    def __init__(self, backbone_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.crop_head = nn.Linear(self.backbone.num_features, NUM_CROPS)
        self.disease_head = nn.Linear(self.backbone.num_features, NUM_DISEASES)

    def forward(self, images):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features)


model = MultiTaskNet(HP["backbone"]).to(DEVICE)


@torch.no_grad()
def predict(images):
    model.eval()
    crop_logits, disease_logits = model(images.to(DEVICE))
    crop_log_prob = F.log_softmax(crop_logits.float(), dim=1)
    disease_log_prob = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log_prob[:, pairs[:, 0]] + disease_log_prob[:, pairs[:, 1]]

# 4. Training

In [4]:
optimizer = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=HP["label_smoothing"])
best_val, best_state = -1.0, None

for epoch in range(EPOCHS):
    model.train()
    for images, crop_labels, disease_labels, _ in train_loader:
        images = images.to(DEVICE)
        crop_labels, disease_labels = crop_labels.to(DEVICE), disease_labels.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()):
            crop_logits, disease_logits = model(images)
            crop_loss = criterion(crop_logits, crop_labels)
            disease_loss = criterion(disease_logits, disease_labels)
            loss = crop_loss + HP["disease_loss_weight"] * disease_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    scheduler.step()
    val_top1 = val_accuracy(predict, val_loader)
    print(f"epoch {epoch + 1}: crop loss {crop_loss.item():.3f}, "
          f"disease loss {disease_loss.item():.3f}, val top-1 {val_top1:.2f}%")
    if val_top1 > best_val:  # keep the best epoch by val, never by test
        best_val, best_state = val_top1, copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)

epoch 1: crop loss 4.015, disease loss 5.105, val top-1 2.00%


<All keys matched successfully>

# 5. Evaluation

In [5]:
results = evaluate(predict, seen_loader, unseen_loader, RUN_NAME, HP, robust=True)

{
  "model": "baseline2_multitask_debug",
  "seen_top1": 1.0,
  "seen_top5": 3.0,
  "unseen_top1": 2.0,
  "unseen_top5": 8.0,
  "harmonic_mean": 1.33,
  "crop_acc": 9.0,
  "disease_acc": 6.0,
  "robustness": {
    "brightness": {
      "top1": 1.0,
      "drop": 0.0
    },
    "blur": {
      "top1": 0.0,
      "drop": 1.0
    },
    "noise": {
      "top1": 1.0,
      "drop": 0.0
    }
  }
}


# 6. Save

In [6]:
CHECKPOINT_DIR.mkdir(exist_ok=True)
torch.save({"model_name": RUN_NAME, "state_dict": model.state_dict(), "hp": HP},
           CHECKPOINT_DIR / f"{RUN_NAME}.pt")